# 08 · Topics, Subscriptions & Filters

**Topics** are like queues from the sender's perspective — you call `SendMessageAsync` exactly the same way. The difference is on the receive side: any number of **subscriptions** can be attached to the topic, and each subscription gets its own copy of every message that matches its filter.

Our Bicep deploys `demo-topic` with three subscriptions:

| Subscription | Filter |
|---|---|
| `all` | (none — matches everything) |
| `high-priority` | `SqlFilter`: `priority = 'high'` |
| `orders` | `CorrelationFilter`: `label = 'order'` (i.e. `Subject = "order"`) |


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.TopicName);

## 1. Send messages with various properties

In [ ]:
await sender.SendMessagesAsync(new[]
{
    new ServiceBusMessage("low-priority log")  { ApplicationProperties = { ["priority"] = "low"  } },
    new ServiceBusMessage("urgent alert")      { ApplicationProperties = { ["priority"] = "high" } },
    new ServiceBusMessage("order #42") { Subject = "order", ApplicationProperties = { ["priority"] = "normal" } },
    new ServiceBusMessage("order #43") { Subject = "order", ApplicationProperties = { ["priority"] = "high" } },
});
Console.WriteLine("Sent 4 messages to the topic.");

## 2. Read each subscription

In [ ]:
async Task DrainAsync(string subscription)
{
    var receiver = client.CreateReceiver(Config.TopicName, subscription, new ServiceBusReceiverOptions
    {
        ReceiveMode = ServiceBusReceiveMode.ReceiveAndDelete
    });

    Console.WriteLine($"\n--- subscription: {subscription} ---");
    while (true)
    {
        var m = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(2));
        if (m is null) break;
        var pri = m.ApplicationProperties.TryGetValue("priority", out var v) ? v : "?";
        Console.WriteLine($"  body={m.Body}  subject={m.Subject ?? "-"}  priority={pri}");
    }
    await receiver.DisposeAsync();
}

await DrainAsync("all");
await DrainAsync("high-priority");
await DrainAsync("orders");

## 3. Filter types

| Type | Matches by | Example |
|---|---|---|
| `SqlFilter` | SQL-92-like expression over system + app properties | `priority = 'high' AND region = 'us-east'` |
| `CorrelationFilter` | Exact-match on a few system fields (cheaper) | `Label = 'order'`, `MessageId = '...'`, `To = '...'` |
| `TrueFilter` / `FalseFilter` | Catch-all / drop-all | Used as defaults |

**Prefer `CorrelationFilter`** when you can — it's evaluated more efficiently than SQL filters.

A subscription can have **multiple rules**; a message matches the subscription if **any** rule matches. You manage rules with the Administration client.

## 4. Add a rule at runtime

In [ ]:
using Azure.Messaging.ServiceBus.Administration;

var admin = new ServiceBusAdministrationClient(Config.ConnectionString);

// Remove a rule if it already exists, then re-create it (idempotent for demos)
try { await admin.DeleteRuleAsync(Config.TopicName, "all", "us-east-only"); } catch { }
await admin.CreateRuleAsync(Config.TopicName, "all",
    new CreateRuleOptions("us-east-only", new SqlRuleFilter("region = 'us-east'")));

await foreach (var r in admin.GetRulesAsync(Config.TopicName, "all"))
    Console.WriteLine($"rule: {r.Name}  filter={r.Filter}");

In [ ]:
await sender.DisposeAsync();
await client.DisposeAsync();

Next: [`09-transactions.ipynb`](09-transactions.ipynb)